# 05_Unsupervised_Feature_Engineering

In [5]:
# To Do: AI Use Disclaimer

In [6]:
# !pip install contractions

In [7]:
import pandas as pd
import duckdb
import polars as pl
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import contractions

In [8]:
# Load parquet into lazy dataframe and display the head
PARQUET_PATH = '../data/processed/cleaned_complaints.parquet'
df_cleaned = pl.scan_parquet(PARQUET_PATH)

# print(f"Total number of records: {df.select(pl.len()).collect()}")
# df.head(5).collect()

print(df_cleaned.schema)
print(df_cleaned.select(pl.len()).collect())
print(df_cleaned.head(5).collect())

C:\Users\Tony\AppData\Local\Temp\ipykernel_8168\2523851684.py:8: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(df_cleaned.schema)


Schema({'Date received': Date, 'Product': String, 'Sub-product': String, 'Issue': String, 'Sub-issue': String, 'Consumer complaint narrative': String, 'Company public response': String, 'Company': String, 'State': String, 'ZIP code': String, 'Tags': String, 'Consumer consent provided?': String, 'Submitted via': String, 'Date sent to company': Date, 'Company response to consumer': String, 'Timely response?': Boolean, 'Consumer disputed?': String, 'Complaint ID': Int64, 'cleaned_consumer_narrative': String})
shape: (1, 1)
┌────────┐
│ len    │
│ ---    │
│ u32    │
╞════════╡
│ 806512 │
└────────┘
shape: (5, 19)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Date      ┆ Product   ┆ Sub-produ ┆ Issue     ┆ … ┆ Timely    ┆ Consumer  ┆ Complaint ┆ cleaned_ │
│ received  ┆ ---       ┆ ct        ┆ ---       ┆   ┆ response? ┆ disputed? ┆ ID        ┆ consumer │
│ ---       ┆ str       ┆ ---       ┆ str       ┆   ┆ ---       ┆ ---       ┆ -

In [9]:
# Load non-null narratives into pandas
df_narratives = (
    df_cleaned
    .filter(pl.col('cleaned_consumer_narrative') != '[No Narrative]')
    .collect()
    .to_pandas()
)

print(df_narratives.shape)
df_narratives.head()

(419894, 19)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,cleaned_consumer_narrative
0,2019-11-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,XXXX claimed they delivered a package to my ad...,NaN,DISCOVER BANK,MA,021XX,NaN,Consent provided,Web,2019-11-18,Closed with explanation,True,N/A,3442136,REDACTED claimed they delivered a package to m...
1,2020-04-10,Credit card or prepaid card,General-purpose prepaid card,Trouble using the card,Trouble getting information about the card,I got a Brinks Money pre-paid card in the mail...,Company has responded to the consumer and the ...,Netspend Corporation,IL,60657,NaN,Consent provided,Web,2020-04-14,Closed with explanation,True,N/A,3601853,I got a Brinks Money pre-paid card in the mail...
2,2019-07-09,Credit card or prepaid card,Store credit card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,On XX/XX/XXXX I was called by a creditor Nelso...,NaN,Nelson Cruz & Associates LLC,TN,37043,NaN,Consent provided,Web,2019-07-09,Closed with explanation,True,N/A,3300820,On REDACTED_DATE I was called by a creditor Ne...
3,2020-07-10,Credit card or prepaid card,General-purpose credit card or charge card,Trouble using your card,Can't use card to make purchases,Around XX/XX/2020 i XXXX XXXX XXXX opened a cr...,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,937XX,NaN,Consent provided,Web,2020-07-10,Closed with explanation,True,N/A,3739698,Around REDACTED / REDACTED /2020 i REDACTED RE...
4,2019-06-24,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,Equifax sent me credit card suggestions to hel...,NaN,"EQUIFAX, INC.",OR,971XX,NaN,Consent provided,Web,2019-06-24,Closed with explanation,True,N/A,3285243,Equifax sent me credit card suggestions to hel...


In [10]:
# Check for camel case (e.g., advisorDollar, creditDispute)
pattern = r'\b[a-z]{5,}[A-Z][a-z]{5,}'

concat_count = (
    df_narratives['cleaned_consumer_narrative']
    .str.contains(pattern, regex=True, na=False)
    .sum()
)

total = len(df_narratives)

print(f'Likely concatenated records: {concat_count:,}')
print(f'Percent affected: {concat_count / total:.2%}')

Likely concatenated records: 237
Percent affected: 0.06%


In [11]:
# Inspect random sample of camel case hits
df_narratives['camel_case_fragments'] = (
    df_narratives['cleaned_consumer_narrative']
    .str.findall(pattern)
)

df_narratives.loc[
    df_narratives['camel_case_fragments'].str.len() > 0,
    ['cleaned_consumer_narrative', 'camel_case_fragments']
].sample(
    n=10,
    random_state=42
)

,cleaned_consumer_narrative,camel_case_fragments
207466,REDACTED REDACTED From : REDACTED REDACTED RED...,"[closedClosed, closedClosed, closedClosed, clo..."
23238,Company wrote off debt w governmentSec.gov. Fo...,[creditCompany]
385915,This is a complaint against Shellpoint Mortgag...,"[monthProperty, monthProperty]"
232738,"b'Today, I, acting as Attorney-in-Fact for Rob...",[informationSocial]
6368,"Capital, one has put, through REDACTED to the ...",[noissueCapital]
319592,I feel compelled to share my deeply dishearten...,[facingAmerican]
9091,I submitted a detailed dispute to Capital One ...,[failuresCapital]
396673,"To Whom It May Concern, I am writing to submit...",[informationCitizens]
203332,"Dear Sir/Madam, I am filing a complaint agains...",[appliancesPennymac]
396647,I applied and received a capital one platinum ...,[smallCredit]


In [14]:
# Random sample of narratives <30 chars
short_narratives = df_narratives[
    df_narratives['cleaned_consumer_narrative'].str.len() < 30
]

print(f'Narratives under 30 characters: {len(short_narratives):,}')

short_narratives[
    ['cleaned_consumer_narrative']
].sample(
    n=min(10, len(short_narratives)),
    random_state=41
)

Narratives under 30 characters: 259


,cleaned_consumer_narrative
56452,Excessive overdraft fees
23650,see attachemnt
318147,See the attached documents.
228399,Open with consent
317320,I didt apply for this card
274879,REDACTED can't link online
158050,Chase bank scammed me.
264879,never used no cards
109057,Someone stole my identity.
418751,I CANT ACCESS ACCOUNT.


In [7]:
# Filter out the short narratives
before_count = len(df_narratives)

df_narratives = df_narratives[
    df_narratives['cleaned_consumer_narrative'].str.len() >= 30
].copy()

after_count = len(df_narratives)

print(f'Before filter: {before_count:,}')
print(f'After filter: {after_count:,}')
print(f'Records removed: {before_count - after_count:,}')

Before filter: 419,894
After filter: 419,635
Records removed: 259


In [8]:
# # Run once if needed
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

def preprocess_narratives(
    df,
    text_col='cleaned_consumer_narrative',
    output_col='processed_narrative'
):
    """
    Lowercase, tokenize, remove stopwords, and lemmatize narrative text.
    Returns a copy of df with a processed text column.
    """

    custom_stopwords = {
        'redacted',
        'redacted_date'
    }

    stop_words = set(stopwords.words('english')) | custom_stopwords

    # Lemmatize - assume every word is a noun.
    # Will not change verbs, e.g., "delivered" will still be "delivered".
    lemmatizer = WordNetLemmatizer()

    def preprocess_text(text):
        # Lowercase
        text = text.lower()

        # Expand contractions: didn't -> did not
        text = contractions.fix(text)

        # Tokenize using regex: keeps alphabetic words only
        tokens = re.findall(r'\b[a-z]+\b', text)

        # Remove stopwords and short tokens
        tokens = [
            token for token in tokens
            if token not in stop_words and len(token) >= 3
        ]

        # Lemmatize
        tokens = [lemmatizer.lemmatize(token) for token in tokens]

        # Return processed text as a string for vectorizers
        return ' '.join(tokens)

    # Apply the function to the text column in the dataframe
    df_out = df.copy()

    df_out[output_col] = (
        df_out[text_col]
        .fillna('')
        .apply(preprocess_text)
    )

    return df_out

In [9]:
# Process the text, including lower-casing, lemmatizing, stop word removal, and tokenizing
df_narratives = preprocess_narratives(df_narratives)

df_narratives[
    ['cleaned_consumer_narrative', 'processed_narrative']
].head()

,cleaned_consumer_narrative,processed_narrative
0,REDACTED claimed they delivered a package to m...,claimed delivered package address never receiv...
1,I got a Brinks Money pre-paid card in the mail...,got brink money pre paid card mail assuming un...
2,On REDACTED_DATE I was called by a creditor Ne...,called creditor nelson cruz associate claimed ...
3,Around REDACTED / REDACTED /2020 i REDACTED RE...,around opened credit card account online capit...
4,Equifax sent me credit card suggestions to hel...,equifax sent credit card suggestion help impro...


In [10]:
# Drop camel case column before exporting
df_narratives = df_narratives.drop(
    columns=['camel_case_fragments']
)

In [11]:
df_narratives.info()

<class 'pandas.DataFrame'>
Index: 419635 entries, 0 to 419893
Data columns (total 20 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   Date received                 419635 non-null  datetime64[ms]
 1   Product                       419635 non-null  str           
 2   Sub-product                   419635 non-null  str           
 3   Issue                         419635 non-null  str           
 4   Sub-issue                     419635 non-null  str           
 5   Consumer complaint narrative  419635 non-null  str           
 6   Company public response       202784 non-null  str           
 7   Company                       419635 non-null  str           
 8   State                         419635 non-null  str           
 9   ZIP code                      419635 non-null  str           
 10  Tags                          84317 non-null   str           
 11  Consumer consent provided?   

In [12]:
OUTPUT_PATH = '../data/processed/processed_narratives.parquet'

df_narratives.to_parquet(
    OUTPUT_PATH,
    index=False
)